# Injection d'erreurs — Titanic (niveau Moyen)
## Projet : Vers une évaluation fiable des workflows de data preparation générés par LLM

**Dataset :** titanic-dataset.csv (Kaggle, yasserh)
**Niveau :** Moyen uniquement (pour l'instant — low/high pourront être ajoutés plus tard)

---

### Colonnes du dataset (standard Kaggle titanic)

`PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked`

### Adaptation par rapport à hotel_bookings

Titanic n'a **pas de colonne date** — la famille "format_errors" (dates) n'est donc pas
applicable ici. On garde 3 familles : missing values, outliers, typos (texte + numérique).

### Répartition des colonnes (medium, sans conflit)

| Famille | Colonnes | Taux |
|---|---|---|
| missing values | `Age`, `Embarked`, `Cabin` | 10% |
| outliers | `Age`, `Fare`, `SibSp`, `Parch` | 2% |
| typos texte | `Sex`, `Embarked` | 8% |
| typos numérique | `Fare`, `Age` |  8% |

⚠️ **Vérifie que les noms de colonnes ci-dessus correspondent exactement à ton CSV**
(`df_clean.columns.tolist()` dans la cellule suivante) avant de lancer l'injection —
certains exports Kaggle utilisent des variantes de casse (`age` vs `Age`).

## 1. Imports et chargement du dataset clean

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

DATA_DIR = Path("../datasets/titanic")
DATA_DIR.mkdir(parents=True, exist_ok=True)

df_clean = pd.read_csv(DATA_DIR / "clean.csv", low_memory=False)
print(f"Dimensions du dataset clean : {df_clean.shape}")
print(f"Colonnes : {df_clean.columns.tolist()}")
df_clean.head()

Dimensions du dataset clean : (891, 12)
Colonnes : ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Les fonctions d'injection (identiques à hotel_bookings)

In [2]:
def inject_format_errors(df, columns=None, error_rate=0.15, random_state=42):
    '''
    Injecte des incohérences de FORMAT sur des colonnes de type date.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset propre (version de référence).
    columns : list[str] or None
        Colonnes de type date à corrompre. Par défaut : ['reservation_status_date'].
    error_rate : float
        Proportion de cellules à corrompre par colonne (0 à 1).
    random_state : int
        Graine pour la reproductibilité.

    Returns
    -------
    df_noisy : pd.DataFrame
        Copie du dataset avec les formats de date corrompus.
    error_log : pd.DataFrame
        Journal des erreurs injectées : row_index, column, original_value,
        injected_value, error_type.
    '''
    rng = np.random.default_rng(random_state)
    df_noisy = df.copy()
    columns = columns or ["reservation_status_date"]
    log_records = []

    def reformat(date_str):
        try:
            d = pd.to_datetime(date_str)
        except Exception:
            return date_str, "unparsable_original"

        variants = [
            d.strftime("%d/%m/%Y"),          # 23/12/2015
            d.strftime("%m/%d/%Y"),          # 12/23/2015 (format US, ambigu)
            d.strftime("%B %d, %Y"),         # December 23, 2015
            d.strftime("%d-%b-%Y"),          # 23-Dec-2015
            d.strftime("%Y/%m/%d"),          # 2015/12/23
            str(int(d.timestamp())),         # timestamp brut
        ]
        choice = rng.integers(0, len(variants))
        return variants[choice], "date_format_inconsistency"

    for col in columns:
        if col not in df_noisy.columns:
            continue
        n_rows = len(df_noisy)
        n_to_corrupt = int(n_rows * error_rate)
        idx_to_corrupt = rng.choice(n_rows, size=n_to_corrupt, replace=False)

        for idx in idx_to_corrupt:
            original_value = df_noisy.at[idx, col]
            new_value, err_type = reformat(original_value)
            df_noisy.at[idx, col] = new_value
            log_records.append({
                "row_index": idx,
                "column": col,
                "original_value": original_value,
                "injected_value": new_value,
                "error_type": err_type,
            })

    error_log = pd.DataFrame(log_records)
    return df_noisy, error_log


def inject_outliers(df, column_bounds=None, error_rate=0.02, random_state=42):
    '''
    Injecte des valeurs ABERRANTES sur des colonnes numériques.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset propre (version de référence).
    column_bounds : dict or None
        Dictionnaire {colonne: (min, max)} définissant la plage des valeurs aberrantes.
        Par défaut, couvre adr, lead_time, adults, babies, stays_in_week_nights,
        days_in_waiting_list.
    error_rate : float
        Proportion de cellules à corrompre par colonne (0 à 1).
    random_state : int
        Graine pour la reproductibilité.

    Returns
    -------
    df_noisy : pd.DataFrame
        Copie du dataset avec des valeurs aberrantes injectées.
    error_log : pd.DataFrame
        Journal des erreurs injectées : row_index, column, original_value,
        injected_value, error_type.
    '''
    rng = np.random.default_rng(random_state)
    df_noisy = df.copy()
    log_records = []

    default_bounds = {
    "adr": (-500, 10000),
    "babies": (10, 50),
    "stays_in_week_nights": (200, 1000),
    "days_in_waiting_list": (2000, 9000),
}
    column_bounds = column_bounds or default_bounds

    for col, (low, high) in column_bounds.items():
        if col not in df_noisy.columns:
            continue
        n_rows = len(df_noisy)
        n_to_corrupt = max(1, int(n_rows * error_rate))
        idx_to_corrupt = rng.choice(n_rows, size=n_to_corrupt, replace=False)

        for idx in idx_to_corrupt:
            original_value = df_noisy.at[idx, col]
            outlier_value = rng.uniform(low, high)
            if pd.api.types.is_integer_dtype(df[col]):
                outlier_value = int(outlier_value)
            else:
                outlier_value = round(float(outlier_value), 2)

            df_noisy.at[idx, col] = outlier_value
            log_records.append({
                "row_index": idx,
                "column": col,
                "original_value": original_value,
                "injected_value": outlier_value,
                "error_type": "outlier_value",
            })

    error_log = pd.DataFrame(log_records)
    return df_noisy, error_log

In [3]:
def inject_missing_values(df, columns=None, error_rate=0.10, random_state=42):
    '''
    Injecte des valeurs MANQUANTES (explicites ou déguisées) sur les colonnes ciblées.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset propre (version de référence).
    columns : list[str] or None
        Colonnes à corrompre. Par défaut : ['country', 'agent', 'children',
        'market_segment', 'meal'].
    error_rate : float
        Proportion de cellules à corrompre par colonne (0 à 1).
    random_state : int
        Graine pour la reproductibilité.

    Returns
    -------
    df_noisy : pd.DataFrame
        Copie du dataset avec des valeurs manquantes (réelles ou déguisées) injectées.
    error_log : pd.DataFrame
        Journal des erreurs injectées : row_index, column, original_value,
        injected_value, error_type.
    '''
    rng = np.random.default_rng(random_state)
    df_noisy = df.copy()
    columns = columns or ["country", "agent", "children", "market_segment", "meal"]
    log_records = []

    # Variantes de "manquant" : (valeur_injectee, label_pour_le_log)
    variants = [
        (np.nan, "missing_nan"),
        ("", "missing_empty_string"),
        ("NA", "missing_NA_string"),
        ("N/A", "missing_N/A_string"),
        ("unknown", "missing_unknown_string"),
        (" ", "missing_whitespace"),
    ]

    for col in columns:
        if col not in df_noisy.columns:
            continue
        # On force la colonne en type 'object' AVANT d'injecter : sinon pandas refuse
        # d'écrire une chaîne ("unknown", "NA"...) dans une colonne numérique (float64/int64).
        df_noisy[col] = df_noisy[col].astype(object)

        n_rows = len(df_noisy)
        n_to_corrupt = int(n_rows * error_rate)
        idx_to_corrupt = rng.choice(n_rows, size=n_to_corrupt, replace=False)

        for idx in idx_to_corrupt:
            original_value = df.at[idx, col]  # valeur ORIGINALE (jamais déjà bruitée)
            choice_idx = rng.integers(0, len(variants))
            new_value, err_type = variants[choice_idx]
            df_noisy.at[idx, col] = new_value
            log_records.append({
                "row_index": idx,
                "column": col,
                "original_value": original_value,
                "injected_value": new_value,
                "error_type": err_type,
            })

    error_log = pd.DataFrame(log_records)
    return df_noisy, error_log


def _typo_text(value, rng):
    '''Applique une faute de frappe aléatoire à une chaîne de caractères.'''
    s = str(value)
    if len(s) < 2:
        return s, "typo_too_short_unchanged"

    op = rng.integers(0, 5)
    i = rng.integers(0, len(s) - 1)

    if op == 0:
        # Lettre supprimée
        new_s = s[:i] + s[i + 1:]
        err_type = "typo_missing_letter"
    elif op == 1:
        # Lettre dupliquée
        new_s = s[:i] + s[i] + s[i:]
        err_type = "typo_duplicated_letter"
    elif op == 2:
        # Deux lettres adjacentes inversées
        new_s = s[:i] + s[i + 1] + s[i] + s[i + 2:]
        err_type = "typo_swapped_letters"
    elif op == 3:
        # Casse aléatoire (ex: Casablanca -> CASAblanca)
        cut = rng.integers(1, len(s))
        new_s = s[:cut].upper() + s[cut:].lower()
        err_type = "typo_random_case"
    else:
        # Espaces parasites en début/fin
        new_s = "  " + s + " "
        err_type = "typo_extra_whitespace"

    return new_s, err_type


def _typo_numeric(value, rng):
    '''Injecte un ou plusieurs caractères non numériques dans une valeur numérique.'''
    s = str(value)
    letter_pool = ["O", "l", "S", "B", "€", "kg", "j"]
    letter = letter_pool[rng.integers(0, len(letter_pool))]

    op = rng.integers(0, 3)
    if op == 0:
        # Remplace un zéro par la lettre "O" (confusion classique zero/O)
        if "0" in s:
            idx0 = s.index("0")
            new_s = s[:idx0] + "O" + s[idx0 + 1:]
        else:
            new_s = s + "O"
        err_type = "typo_zero_as_letter_O"
    elif op == 1:
        # Ajoute une unité/texte à la fin (ex: "75.0" -> "75.0kg")
        new_s = s + letter
        err_type = "typo_unit_suffix_injected"
    else:
        # Insère un caractère au milieu de la valeur
        i = rng.integers(1, max(2, len(s)))
        new_s = s[:i] + letter + s[i:]
        err_type = "typo_letter_inside_number"

    return new_s, err_type


def inject_typos(df, text_columns=None, numeric_columns=None, error_rate=0.10, random_state=42):
    '''
    Injecte des erreurs TYPOGRAPHIQUES : fautes de frappe sur colonnes textuelles,
    et caractères non numériques insérés dans des colonnes numériques.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset propre (version de référence).
    text_columns : list[str] or None
        Colonnes textuelles/catégorielles à corrompre.
        Par défaut : ['hotel', 'country', 'meal', 'deposit_type', 'customer_type'].
    numeric_columns : list[str] or None
        Colonnes numériques dans lesquelles injecter des caractères.
        Par défaut : ['adr', 'lead_time', 'adults'].
    error_rate : float
        Proportion de cellules à corrompre par colonne (0 à 1).
    random_state : int
        Graine pour la reproductibilité.

    Returns
    -------
    df_noisy : pd.DataFrame
        Copie du dataset avec des erreurs typographiques injectées.
        NB : après injection, les colonnes numériques touchées deviennent de type `object`
        (mélange de nombres et de texte), ce qui est volontaire et reproduit un cas réel.
    error_log : pd.DataFrame
        Journal des erreurs injectées : row_index, column, original_value,
        injected_value, error_type.
    '''
    rng = np.random.default_rng(random_state)
    df_noisy = df.copy()
    text_columns = text_columns or ["hotel", "deposit_type", "customer_type"]
    numeric_columns = numeric_columns or ["lead_time", "adults"]
    log_records = []

    def apply_corruption(columns, corrupt_fn):
        for col in columns:
            if col not in df_noisy.columns:
                continue
            # S'assurer que la colonne peut recevoir du texte
            df_noisy[col] = df_noisy[col].astype(object)

            n_rows = len(df_noisy)
            n_to_corrupt = int(n_rows * error_rate)
            idx_to_corrupt = rng.choice(n_rows, size=n_to_corrupt, replace=False)

            for idx in idx_to_corrupt:
                original_value = df.at[idx, col]  # valeur ORIGINALE (pas déjà bruitée)
                if pd.isna(original_value):
                    continue
                new_value, err_type = corrupt_fn(original_value, rng)
                df_noisy.at[idx, col] = new_value
                log_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": original_value,
                    "injected_value": new_value,
                    "error_type": err_type,
                })

    apply_corruption(text_columns, _typo_text)
    apply_corruption(numeric_columns, _typo_numeric)

    error_log = pd.DataFrame(log_records)
    return df_noisy, error_log


DISGUISED_MISSING = {"", "na", "n/a", "unknown", " ", "nan", "none", "null"}

def profile_dataset(df, top_n=5):
    '''
    Construit un profil statistique du dataset, utilisable dans un prompt LLM (approche
    "prompt avec profilage").

    Returns
    -------
    profile : pd.DataFrame
        Une ligne par colonne avec : dtype détecté, % manquants (réels + déguisés),
        statistiques descriptives (numérique) ou valeurs les plus fréquentes (catégorielle).
    '''
    records = []
    n_rows = len(df)

    for col in df.columns:
        series = df[col]
        is_numeric = pd.api.types.is_numeric_dtype(series)

        # Manquants réels
        n_na_real = series.isna().sum()

        # Manquants déguisés (seulement pertinent pour les colonnes non numériques)
        if not is_numeric:
            n_na_disguised = series.astype(str).str.strip().str.lower().isin(DISGUISED_MISSING).sum()
        else:
            n_na_disguised = 0

        pct_missing = round(100 * (n_na_real + n_na_disguised) / n_rows, 2)

        record = {
            "column": col,
            "dtype_detected": "numeric" if is_numeric else "categorical/text",
            "pct_missing": pct_missing,
            "n_unique": series.nunique(dropna=True),
        }

        if is_numeric:
            record["min"] = series.min()
            record["max"] = series.max()
            record["mean"] = round(series.mean(), 2)
            record["std"] = round(series.std(), 2)
        else:
            top_values = series.value_counts().head(top_n).to_dict()
            record["top_values"] = top_values

        records.append(record)

    return pd.DataFrame(records)


## 3. Configuration niveau Moyen (adaptée aux colonnes de Titanic)

In [4]:
MEDIUM_CONFIG = {
    "missing":       {"rate": 0.10, "columns": ["Age", "Embarked", "Cabin"]},
    "outliers":      {"rate": 0.02, "columns": ["Age", "Fare", "SibSp", "Parch"]},
    "typos_text":    {"rate": 0.08, "columns": ["Sex", "Embarked"]},
    "typos_numeric": {"rate": 0.08, "columns": ["Fare", "Age"]},
}

# Bornes réalistes pour la détection/injection d'outliers
OUTLIER_BOUNDS = {
    "Age": (150, 300),          # un age > 150 ans est absurde
    "Fare": (2000, 9000),       # tarif de billet aberrant
    "SibSp": (20, 50),          # nombre de frères/sœurs à bord aberrant
    "Parch": (20, 50),          # nombre de parents/enfants à bord aberrant
}

## 4. `inject_all()` — sans famille format (pas de colonne date)

In [5]:
def inject_all_titanic(df_clean, random_state=42):
    cfg = MEDIUM_CONFIG
    df_current = df_clean.copy()
    all_logs = []

    df_current, log_missing = inject_missing_values(
        df_current, columns=cfg["missing"]["columns"],
        error_rate=cfg["missing"]["rate"], random_state=random_state)
    log_missing["error_family"] = "missing_values"
    all_logs.append(log_missing)

    bounds = {c: OUTLIER_BOUNDS[c] for c in cfg["outliers"]["columns"]}
    df_current, log_outliers = inject_outliers(
        df_current, column_bounds=bounds,
        error_rate=cfg["outliers"]["rate"], random_state=random_state)
    log_outliers["error_family"] = "outliers"
    all_logs.append(log_outliers)

    df_current, log_typos = inject_typos(
        df_current,
        text_columns=cfg["typos_text"]["columns"],
        numeric_columns=cfg["typos_numeric"]["columns"],
        error_rate=cfg["typos_text"]["rate"], random_state=random_state)
    log_typos["error_family"] = "typos"
    all_logs.append(log_typos)

    full_log = pd.concat(all_logs, ignore_index=True)
    full_log["noise_level"] = "medium"
    return df_current, full_log


df_noisy, error_log = inject_all_titanic(df_clean, random_state=42)
print(f"{len(error_log)} erreurs injectées | shape={df_noisy.shape}")
print(error_log.groupby("error_family").size())

602 erreurs injectées | shape=(891, 12)
error_family
missing_values    267
outliers           68
typos             267
dtype: int64


## 5. Sauvegarde

In [6]:
df_noisy.to_csv(DATA_DIR / "noisy_medium.csv", index=False)
error_log.to_csv(DATA_DIR / "error_log_medium.csv", index=False)

metadata = {
    "dataset": "titanic",
    "n_rows": int(df_clean.shape[0]),
    "n_cols": int(df_clean.shape[1]),
    "levels": {
        "medium": {
            "total_errors": int(len(error_log)),
            "error_rate_global": round(len(error_log) / (df_clean.shape[0] * df_clean.shape[1]), 4),
            "errors_by_family": error_log["error_family"].value_counts().to_dict(),
            "config": MEDIUM_CONFIG,
        }
    },
    "note": "Pas de famille 'format_errors' : titanic n'a pas de colonne date. "
            "Niveaux low/high non générés pour l'instant.",
}

with open(DATA_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str, ensure_ascii=False)

print(f"Fichiers sauvegardés dans {DATA_DIR}/ :")
print("  - noisy_medium.csv")
print("  - error_log_medium.csv")
print("  - metadata.json")

Fichiers sauvegardés dans ..\datasets\titanic/ :
  - noisy_medium.csv
  - error_log_medium.csv
  - metadata.json
